# Emotion Binary Classification from Text (Joy vs Sadness)

**Name:** Muhammad Hassan Yousaf  
**Roll Number:** FA24-BSE-082  
**Course:** Machine Learning Fundamentals (AIC354), Assignment 4, Task 1 (Binary)  
**Instructor:** Dr. Rao Muhammad Adeel Nawab  
**University:** COMSATS University Islamabad, Lahore Campus  

**Deployment Link:** [https://spam-detector-hassan.streamlit.app](https://spam-detector-hassan.streamlit.app)  
**GitHub:** [https://github.com/MHassan0000/ML-Prediction-Models](https://github.com/MHassan0000/ML-Prediction-Models)

In this notebook I build a binary emotion classifier that distinguishes between joy and sadness in short text. I use TF-IDF to convert the text into numbers, then compare 6 machine learning algorithms using 10-fold cross validation. Accuracy is used as the evaluation metric since the classes are reasonably balanced.

**6 Algorithms compared:**
1. Random Forest
2. Naive Bayes (MultinomialNB)
3. k-Nearest Neighbors
4. Support Vector Machine (LinearSVC)
5. Logistic Regression
6. Decision Tree

## Step 1: Import Libraries

In [ ]:
import os
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from tqdm import tqdm

pd.set_option('display.float_format', lambda v: f'{v:.4f}')

print(f'NumPy        : {np.__version__}')
print(f'Pandas       : {pd.__version__}')
print(f'Scikit-learn : {sklearn.__version__}')
print(f'Matplotlib   : {matplotlib.__version__}')

## Step 2: Load Dataset

The dataset comes from the dair-ai/emotion collection on HuggingFace. It has 20,000 labelled texts across 6 emotions. For binary classification I only keep joy (label 1) and sadness (label 0).

In [ ]:
DATA_FILE = 'emotion_full.csv'

if not os.path.isfile(DATA_FILE):
    try:
        from google.colab import files as colab_files
        print('Upload emotion_full.csv when the dialog appears...')
        colab_files.upload()
    except ImportError:
        raise FileNotFoundError(f'{DATA_FILE} not found. Put it in the same folder as this notebook.')
else:
    print(f'Found {DATA_FILE}')

df_all = pd.read_csv(DATA_FILE)
print(f'Full dataset loaded: {df_all.shape[0]} rows')

LABEL_MAP = {0: 'sadness', 1: 'joy'}
df = df_all[df_all['label'].isin(LABEL_MAP.keys())].copy().reset_index(drop=True)
df['emotion'] = df['label'].map(LABEL_MAP)

print(f'After filtering to joy and sadness: {len(df)} rows')
df.head(10)

## Step 3: Understand the Data

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')
print('\nClass distribution:')
print(df['emotion'].value_counts())

In [ ]:
print('\nSample texts:')
for emotion in LABEL_MAP.values():
    sample = df[df['emotion'] == emotion]['text'].iloc[0]
    print(f'  {emotion}: "{sample[:90]}"')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['emotion'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#3498db', '#2ecc71'], edgecolor='white')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

df['text_len'] = df['text'].str.len()
for emotion, color in zip(['joy', 'sadness'], ['#2ecc71', '#3498db']):
    axes[1].hist(df[df['emotion'] == emotion]['text_len'], bins=50,
                 alpha=0.6, label=emotion, color=color)
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Number of characters')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 4: Pre-processing

I convert to lowercase and strip out anything that is not a letter or a space. The texts in this dataset are already pretty clean so the main job here is normalizing case.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Before and after cleaning:')
for idx in [0, 5, 10]:
    orig = df.loc[idx, 'text']
    cleaned = clean_text(orig)
    print(f'  BEFORE: "{orig}"')
    print(f'  AFTER:  "{cleaned}"')
    print()

df['text_clean'] = df['text'].apply(clean_text)
print(f'Cleaned {len(df)} texts.')

## Step 5: Feature Extraction

TF-IDF converts each text into a vector of word importance scores. I use word-level unigrams and limit to the top 5000 features. The vectorizer is fitted inside each CV fold so there is no data leakage.

In [ ]:
X_text = df['text_clean'].values
y = df['label'].values

preview_vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 1), max_features=5000)
X_preview = preview_vec.fit_transform(X_text)

print(f'TF-IDF matrix shape: {X_preview.shape}')
print(f'Vocabulary size    : {len(preview_vec.vocabulary_)} words')
print(f'Sample features    : {list(preview_vec.get_feature_names_out()[:15])}')
print(f'Class counts       : sadness={sum(y==0)}, joy={sum(y==1)}')

## Step 6: Define 6 ML Algorithms

In [ ]:
ALGORITHMS = {
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Naive Bayes (MNB)':   MultinomialNB(alpha=1.0),
    'k-NN':                KNeighborsClassifier(n_neighbors=5),
    'SVM (LinearSVC)':     LinearSVC(C=1.0, max_iter=5000, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
}

for i, name in enumerate(ALGORITHMS, 1):
    print(f'  {i}. {name}')

## Step 7: Set Up 10-Fold Cross Validation

In [ ]:
K = 10
kf = KFold(n_splits=K, shuffle=True, random_state=42)

print(f'K = {K}')
print(f'Total samples  : {len(X_text)}')
print(f'Per test fold  : ~{len(X_text) // K}')
print(f'Per train fold : ~{len(X_text) - len(X_text) // K}')

print(f"\n{'Fold':>4}  {'Train':>8}  {'Test':>6}")
for i, (tr, te) in enumerate(kf.split(X_text), 1):
    print(f'{i:>4}  {len(tr):>8}  {len(te):>6}')

## Step 8: Run Cross Validation

For every fold I fit TF-IDF only on the training portion, then transform both train and test. This prevents any test data from leaking into the vocabulary.

In [ ]:
results = {}
pbar = tqdm(total=len(ALGORITHMS) * K, desc='Training', unit='fit')

for algo_name, algo in ALGORITHMS.items():
    fold_accs = []
    for tr_idx, te_idx in kf.split(X_text):
        vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 1), max_features=5000)
        X_tr = vec.fit_transform(X_text[tr_idx])
        X_te = vec.transform(X_text[te_idx])

        m = clone(algo)
        m.fit(X_tr, y[tr_idx])
        fold_accs.append(accuracy_score(y[te_idx], m.predict(X_te)))
        pbar.update(1)

    results[algo_name] = fold_accs

pbar.close()
print('Done!')

## Step 9: Compare Results

### Per-Fold Accuracy

In [ ]:
fold_df = pd.DataFrame(results, index=[f'Fold {i}' for i in range(1, K + 1)])
fold_df

### Summary Table

In [ ]:
summary = pd.DataFrame({
    'Mean Accuracy': {n: np.mean(v) for n, v in results.items()},
    'Std Accuracy':  {n: np.std(v)  for n, v in results.items()},
    'Min Accuracy':  {n: np.min(v)  for n, v in results.items()},
    'Max Accuracy':  {n: np.max(v)  for n, v in results.items()},
}).sort_values('Mean Accuracy', ascending=False)

print('Sorted by Mean Accuracy (higher is better):')
summary

### Charts

In [ ]:
names  = summary.index.tolist()
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c', '#1abc9c']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

m_vals = [summary.loc[n, 'Mean Accuracy'] for n in names]
s_vals = [summary.loc[n, 'Std Accuracy']  for n in names]
bars = axes[0].barh(names, m_vals, xerr=s_vals, color=colors, edgecolor='white', capsize=4)
axes[0].set_xlabel('Mean Accuracy')
axes[0].set_title('Mean Accuracy (higher is better)')
axes[0].invert_yaxis()
axes[0].set_xlim(0.7, 1.0)
for bar, val in zip(bars, m_vals):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontsize=9)

bp = axes[1].boxplot([results[n] for n in names], labels=names, patch_artist=True)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Distribution')
axes[1].tick_params(axis='x', rotation=30)

for n, c in zip(names, colors):
    axes[2].plot(range(1, K + 1), results[n], 'o-', label=n, color=c, linewidth=2)
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Accuracy per Fold')
axes[2].legend(fontsize=8)
axes[2].xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

## Step 10: Observations

In [ ]:
best  = summary.index[0]
worst = summary.index[-1]

print(f'Best  algorithm: {best}')
print(f'  Mean Accuracy = {summary.loc[best, "Mean Accuracy"]:.4f}')
print(f'  Std  Accuracy = {summary.loc[best, "Std Accuracy"]:.4f}')
print()
print(f'Worst algorithm: {worst}')
print(f'  Mean Accuracy = {summary.loc[worst, "Mean Accuracy"]:.4f}')
print()
print(f'{best} gives the best results so I will use it for the final model.')

## Step 11: Train Final Model

Train the best algorithm on all data and save it.

In [ ]:
BEST_NAME = summary.index[0]
print(f'Training {BEST_NAME} on full dataset...')

final_tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1, 1), max_features=5000)
X_all = final_tfidf.fit_transform(X_text)

final_model = clone(ALGORITHMS[BEST_NAME])
final_model.fit(X_all, y)

train_acc = accuracy_score(y, final_model.predict(X_all))
print(f'Training Accuracy: {train_acc:.4f}')

with open('emotion_binary_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
with open('emotion_binary_tfidf.pkl', 'wb') as f:
    pickle.dump(final_tfidf, f)

print('Saved emotion_binary_model.pkl and emotion_binary_tfidf.pkl')

## Step 12: Test Prediction

In [ ]:
with open('emotion_binary_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('emotion_binary_tfidf.pkl', 'rb') as f:
    loaded_tfidf = pickle.load(f)

test_texts = [
    'I am so happy today, everything is going great!',
    'I feel really sad and alone right now',
    'This is the best day of my life, I got promoted!',
    'I cannot stop crying, I lost my best friend',
    'Feeling wonderful and blessed to have such great people around me',
    'I feel so empty and heartbroken after the breakup',
]

print(f'Predictions using {BEST_NAME}:')
print('=' * 65)

cleaned = [clean_text(t) for t in test_texts]
X_test  = loaded_tfidf.transform(cleaned)
preds   = loaded_model.predict(X_test)

for text, pred in zip(test_texts, preds):
    emotion = LABEL_MAP[pred]
    icon    = '😊' if emotion == 'joy' else '😢'
    print(f'  {icon} [{emotion.upper():>7}]  "{text}"')

print('\nModel is ready for deployment!')